03 — Entrenamiento con MLflow Tracking

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

Mounted at /content/drive
✅ Drive montado


In [2]:
%%capture
!pip install lightgbm==4.6.0 optuna scikit-learn pandas numpy mlflow
!pip install fastapi uvicorn nest-asyncio pyngrok evidently joblib matplotlib seaborn
print('OK')

In [3]:
import json, os, time
import pandas as pd
import numpy as np
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from mlflow.models.signature import infer_signature
from datetime import datetime

BASE_DRIVE = '/content/drive/MyDrive/1. MAESTRÍA CIENCIA DE DATOS/ENTRENAMIENTO PRÁCTICO'
config_path = f'{BASE_DRIVE}/pipeline_config.json'
if os.path.exists(config_path):
    with open(config_path) as f: CONFIG = json.load(f)
    print('✅ Config cargada')
else:
    CONFIG = {
        'dir_raw_data'     : f'{BASE_DRIVE}/Data Cruda',
        'dir_training_data': f'{BASE_DRIVE}/Data Entrenamiento',
        'dir_processed'    : f'{BASE_DRIVE}/Data Preprocesada',
        'train_path'       : f'{BASE_DRIVE}/Data Preprocesada/training_data/preprocessed/train_vars_extrac.csv',
        'test_path'        : f'{BASE_DRIVE}/Data Preprocesada/training_data/preprocessed/test_vars_extrac.csv',
        'model_dir'        : f'{BASE_DRIVE}/Modelo',
        'model_path'       : f'{BASE_DRIVE}/Modelo/lgbm_model.pkl',
        'metadata_path'    : f'{BASE_DRIVE}/Modelo/lgbm_metadata.json',
        'mlflow_uri'       : f'{BASE_DRIVE}/mlruns',
        'experiment_name'  : 'campana_credito_propension',
        'monitor_dir'      : f'{BASE_DRIVE}/monitoreo',
        'target_col'       : 'target',
        'id_cols'          : ['partition','key_value','codunicocli',
                              'fch_creacion','p_fecinformacion','seg_un'],
        'score_col'        : 'prob_value_contact',
    }
    print('⚠️ Config definida localmente')

os.makedirs(CONFIG['model_dir'], exist_ok=True)
mlflow.set_tracking_uri(CONFIG['mlflow_uri'])
mlflow.set_experiment(CONFIG['experiment_name'])
TARGET_COL = CONFIG['target_col']
print(f'✅ MLflow listo')

✅ Config cargada


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


✅ MLflow listo


In [4]:
# Cargar datos preprocesados
if not os.path.exists(CONFIG['train_path']):
    raise FileNotFoundError(
        f"No existe: {CONFIG['train_path']}\n"
        "Ejecuta primero 02_preprocesamiento.ipynb"
    )

train_df = pd.read_csv(CONFIG['train_path'])
test_df  = pd.read_csv(CONFIG['test_path'])
X_train  = train_df.drop(columns=[TARGET_COL])
y_train  = train_df[TARGET_COL]
X_test   = test_df.drop(columns=[TARGET_COL])
y_test   = test_df[TARGET_COL]
print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')

✅ Train: (1059077, 60) | Test: (521636, 60)


In [5]:
# Hiperparámetros base (del lgbm_metadata.json entregado)
PARAMS_BASE = {
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 100,
    'learning_rate'    : 0.1,
    'max_depth'        : -1,
    'num_leaves'       : 31,
    'min_child_samples': 20,
    'min_child_weight' : 0.001,
    'colsample_bytree' : 1.0,
    'subsample'        : 1.0,
    'reg_alpha'        : 0.0,
    'reg_lambda'       : 0.0,
    'random_state'     : 42,
    'n_jobs'           : -1,
    'class_weight'     : 'balanced',
}

with mlflow.start_run(run_name='03_lgbm_base') as run:
    mlflow.log_params(PARAMS_BASE)
    mlflow.log_param('model_type', 'LightGBM')
    mlflow.log_param('timestamp',  datetime.now().strftime('%Y-%m-%d %H:%M'))

    t0 = time.time()
    model = lgb.LGBMClassifier(**PARAMS_BASE)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(20, verbose=False),
                   lgb.log_evaluation(period=20)]
    )
    elapsed = time.time() - t0

    y_tr_proba = model.predict_proba(X_train)[:, 1]
    y_te_proba = model.predict_proba(X_test)[:, 1]
    y_te_pred  = model.predict(X_test)

    auc_train = roc_auc_score(y_train, y_tr_proba)
    auc_test  = roc_auc_score(y_test,  y_te_proba)
    decay     = (auc_train - auc_test) / auc_train * 100
    f1        = f1_score(y_test, y_te_pred)

    mlflow.log_metric('auc_train',          auc_train)
    mlflow.log_metric('auc_test',           auc_test)
    mlflow.log_metric('decay_pct',          decay)
    mlflow.log_metric('f1_score',           f1)
    mlflow.log_metric('training_time_secs', elapsed)

    # Guardar modelo
    joblib.dump(model, CONFIG['model_path'])
    sig = infer_signature(X_train, y_te_proba)
    mlflow.lightgbm.log_model(
        model, 'lgbm_base', signature=sig,
        registered_model_name='LightGBM_Propension_Credito'
    )

    # Guardar metadata
    meta = {
        'ml_name': 'lgbm_base',
        'performance': {
            'auc_train': auc_train, 'auc_test': auc_test,
            'decay_percent': decay, 'f1_score': f1,
            'training_time_segs': elapsed,
        },
        'hyperparameters': PARAMS_BASE,
        'mlflow_run_id': run.info.run_id,
        'timestamp': datetime.now().isoformat(),
    }
    with open(CONFIG['metadata_path'], 'w') as mf:
        json.dump(meta, mf, indent=4)
    mlflow.log_artifact(CONFIG['metadata_path'])
    BASE_RUN_ID = run.info.run_id

print(f'\n✅ ENTRENAMIENTO BASE COMPLETADO')
print(f'   AUC Train: {auc_train:.4f} | AUC Test: {auc_test:.4f} | Decay: {decay:.2f}%')
print(f'   F1: {f1:.4f} | Tiempo: {elapsed:.1f}s')
print(f'   Run ID: {BASE_RUN_ID}')
print('   ➡️  Siguiente: 04_hpo_mlflow.ipynb')

[LightGBM] [Info] Number of positive: 33815, number of negative: 1025262
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.406906 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9397
[LightGBM] [Info] Number of data points in the train set: 1059077, number of used features: 60
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[20]	valid_0's binary_logloss: 0.509893
[40]	valid_0's binary_logloss: 0.486432
[60]	valid_0's binary_logloss: 0.478875
[80]	valid_0's binary_logloss: 0.474248
[100]	valid_0's binary_logloss: 0.470725


2026/05/01 01:15:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 01:15:54 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_model_registry/utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)
Successfully registered model 'LightGBM_Propension_Credit


✅ ENTRENAMIENTO BASE COMPLETADO
   AUC Train: 0.8675 | AUC Test: 0.8539 | Decay: 1.58%
   F1: 0.1749 | Tiempo: 50.7s
   Run ID: 2850ae5a561842b59b2226131a9f711a
   ➡️  Siguiente: 04_hpo_mlflow.ipynb
